In [ ]:
from kafka import KafkaConsumer
import xml.etree.ElementTree as ET
import csv

BROKER = 'cld-kfk-04.brusnika.ltd'
TOPIC = 'bru.revit-plugin.logs'
CSV_FILE = 'plugins_usage.csv'

#URI
NS_URI = 'http://schema.brusnika.tech/mdm/oa/pc/design/revit-plugin/revit-plugin-logs'
#тэги в XML
TAGS = {
    'event': 'event',
    'username': 'username',
    'is_success': 'is-success',            # mapping: is-success -> is_success
    'assembly_timestamp': 'assembly-timestamp',
    'timestamp': 'timestamp'
}

def get_text(elem, tag):
    child = elem.find(f'{{{NS_URI}}}{tag}')
    return child.text if child is not None else ''

consumer = KafkaConsumer(
    TOPIC,
    bootstrap_servers=[BROKER],
    auto_offset_reset='earliest',
    enable_auto_commit=False,
    value_deserializer=lambda m: m.decode('utf-8', errors='ignore'),
    consumer_timeout_ms=10000  # остановится если нет сообщений
)

with open(CSV_FILE, 'w', newline='', encoding='utf-8') as f:
    writer = csv.writer(f)
    writer.writerow(['event','username','is_success','assembly_timestamp','timestamp'])

    for msg in consumer:
        xml = msg.value.strip()
        try:
            root = ET.fromstring(xml)
        except ET.ParseError:
            # можно логировать ошибку и продолжать
            continue

        row = []
        for col, tag in TAGS.items():
            row.append(get_text(root, tag))
        writer.writerow(row)

consumer.close()

In [ ]:
import pandas as pd
from datetime import datetime

st_date = datetime(2026,3,17)
end_date = datetime(2026,3,24)
users = [
            "a.akulinushkina"
            ,"a.filimonov"
            ,"d.usov"
            ,"e.zapretilin"
            ,"i.neustroeva"
            ,"m.bylinkin"
            ,"p.desyatova"
            ,"y.kucher"
            ,"yu.kremenetskaya"
            ,"d.danovskaya"
            ,"u.kartasheva"
            ,"v.semenovych"
            ,"d.shljapenkova"
            ,"BIM"
            ,"n.khabarov"
            ,"an.goncharuk"
            ,"m.andreichenko"
            ,"a.barbakadze"
            ,"m.nosikov"
            ,"d.ishmatova"
            ,"o.erokhova"
            ,'l.gulkova'
            ,'a.mikhailovskii'
            ,'o.khramtsova'
            ]
regex_pattern = '|'.join(users)

format = f"%Y-%m-%d"
df = pd.read_csv('plugins_usage.csv')

df['assembly_timestamp'] = df['assembly_timestamp'].apply(lambda x: datetime.strptime(x[:10],format))
df['timestamp'] = df['timestamp'].apply(lambda x: datetime.strptime(x[:10],format))
df = df[df['timestamp'].between(st_date,end_date)]
df = df[df['username'].str.contains(regex_pattern, case=False, regex=True) == False]


# df.to_excel('plugins_usage_filtered.xlsx',sheet_name='Лист1',index=False)
# print('Сохранено')

In [ ]:
import numpy as np

df = df[df['event'].isna() == False]
df['Блок плагинов'] = df['event'].str.split('_').apply(lambda x: x[0])
df['Блок плагинов'] = np.where(df['Блок плагинов'].str.contains("Префаб") == True,"Префаб",df['Блок плагинов'])
df['Блок плагинов'] = np.where(df['Блок плагинов'].str.contains("МП") == True,"Мастерплан",df['Блок плагинов'])
df['Неделя'] = df['timestamp'].dt.isocalendar().week
df = df[df['Блок плагинов'].isin(["ConsoleTest","Параметры",'Установочник']) == False]

plug_usage = df.groupby('Блок плагинов',as_index=False).agg(Кол_во_запусков=('event','count')
                                                            ,Кол_во_человек=('username','nunique')).sort_values('Кол_во_запусков',ascending=False)
plug_usage = plug_usage.rename({"Кол_во_запусков":"Количество запусков"
,"Кол_во_человек":"Количество человек"}
,axis=1)

In [ ]:
import seaborn as sns
from matplotlib import pyplot as plt
import numpy as np

# Создаем фигуру с белым фоном
fig, ax = plt.subplots(figsize=(12, 8), facecolor='white',dpi=600)
ax.set_facecolor('white')

bar_color = '#C1DC8B'

# Создаем barplot с зеленым цветом как на скриншоте
sns.barplot(
    data=plug_usage,
    y='Блок плагинов',
    x='Количество запусков',
    orient='h',
    ax=ax,
    color=bar_color  # Зеленый цвет как на скриншоте (seagreen)
)

# Добавляем подписи на бары (серые)
for container in ax.containers:
    ax.bar_label(
        container,
        fmt='%.0f',
        padding=5,
        color='#666666',  # Серый цвет подписей
        fontsize=10,
        fontweight='normal'
    )

# Добавляем вертикальные серые линии (сетку)
ax.xaxis.grid(
    True,
    which='major',
    color='#E0E0E0',  # Светло-серый цвет линий
    linestyle='-',
    linewidth=0.5,
    alpha=0.7
)
ax.set_axisbelow(True)  # Сетка под барами

# Настраиваем оси
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['left'].set_color('#CCCCCC')
ax.spines['bottom'].set_color('#CCCCCC')

# Добавляем название графика
ax.set_title(
    'Распределение запусков по блокам плагинов',
    fontsize=16,
    fontweight='bold',
    pad=20,
    color='#333333'
)

# Настраиваем подписи осей
ax.set_xlabel('Количество запусков', fontsize=12, color='#555555', labelpad=10)
ax.set_ylabel('Блок плагинов', fontsize=12, color='#555555', labelpad=10)

# Настраиваем тики
ax.tick_params(axis='both', colors='#666666', labelsize=10)

# Автоматическая подгонка layout
plt.tight_layout()

# Показываем график
plt.show()

In [ ]:
plug_usage = plug_usage.sort_values('Количество человек',ascending=False)
plug_usage

In [ ]:
import seaborn as sns
from matplotlib import pyplot as plt
import numpy as np

# Создаем фигуру с белым фоном
fig, ax = plt.subplots(figsize=(12, 8), facecolor='white',dpi=600)
ax.set_facecolor('white')

bar_color = '#C1DC8B'

# Создаем barplot с зеленым цветом как на скриншоте
sns.barplot(
    data=plug_usage,
    y='Блок плагинов',
    x='Количество человек',
    orient='h',
    ax=ax,
    color=bar_color  # Зеленый цвет как на скриншоте (seagreen)
)

# Добавляем подписи на бары (серые)
for container in ax.containers:
    ax.bar_label(
        container,
        fmt='%.0f',
        padding=5,
        color='#666666',  # Серый цвет подписей
        fontsize=10,
        fontweight='normal'
    )

# Добавляем вертикальные серые линии (сетку)
ax.xaxis.grid(
    True,
    which='major',
    color='#E0E0E0',  # Светло-серый цвет линий
    linestyle='-',
    linewidth=0.5,
    alpha=0.7
)
ax.set_axisbelow(True)  # Сетка под барами

# Настраиваем оси
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['left'].set_color('#CCCCCC')
ax.spines['bottom'].set_color('#CCCCCC')

# Добавляем название графика
ax.set_title(
    'Распределение запусков по блокам плагинов',
    fontsize=16,
    fontweight='bold',
    pad=20,
    color='#333333'
)

# Настраиваем подписи осей
ax.set_xlabel('Количество человек', fontsize=12, color='#555555', labelpad=10)
ax.set_ylabel('Блок плагинов', fontsize=12, color='#555555', labelpad=10)

# Настраиваем тики
ax.tick_params(axis='both', colors='#666666', labelsize=10)

# Автоматическая подгонка layout
plt.tight_layout()

# Показываем график
plt.show()

In [ ]:
user_usage = df.groupby('username',as_index=False).agg(Кол_во_запусков=('event','count')).sort_values('Кол_во_запусков',ascending=False)
user_usage = user_usage.rename({"Кол_во_запусков":"Количество запусков"},axis=1)
user_usage

In [ ]:
ssk = df[df['Блок плагинов'] == "ССК"]

plug_usage = ssk.groupby('event',as_index=False).agg(Кол_во_запусков=('event','count')).sort_values('Кол_во_запусков',ascending=False)
plug_usage = plug_usage.rename({"Кол_во_запусков":"Количество запусков"},axis=1)
plug_usage

In [ ]:
import seaborn as sns
from matplotlib import pyplot as plt
import numpy as np

# Создаем фигуру с белым фоном
fig, ax = plt.subplots(figsize=(12, 8), facecolor='white',dpi=600)
ax.set_facecolor('white')

bar_color = '#C1DC8B'

# Создаем barplot с зеленым цветом как на скриншоте
sns.barplot(
    data=plug_usage,
    y='event',
    x='Количество запусков',
    orient='h',
    ax=ax,
    color=bar_color  # Зеленый цвет как на скриншоте (seagreen)
)

# Добавляем подписи на бары (серые)
for container in ax.containers:
    ax.bar_label(
        container,
        fmt='%.0f',
        padding=5,
        color='#666666',  # Серый цвет подписей
        fontsize=10,
        fontweight='normal'
    )

# Добавляем вертикальные серые линии (сетку)
ax.xaxis.grid(
    True,
    which='major',
    color='#E0E0E0',  # Светло-серый цвет линий
    linestyle='-',
    linewidth=0.5,
    alpha=0.7
)
ax.set_axisbelow(True)  # Сетка под барами

# Настраиваем оси
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['left'].set_color('#CCCCCC')
ax.spines['bottom'].set_color('#CCCCCC')

# Добавляем название графика
ax.set_title(
    'Распределение запусков по блокам плагинов',
    fontsize=16,
    fontweight='bold',
    pad=20,
    color='#333333'
)

# Настраиваем подписи осей
ax.set_xlabel('Количество запусков', fontsize=12, color='#555555', labelpad=10)
ax.set_ylabel('Команда', fontsize=12, color='#555555', labelpad=10)

# Настраиваем тики
ax.tick_params(axis='both', colors='#666666', labelsize=10)

# Автоматическая подгонка layout
plt.tight_layout()

# Показываем график
plt.show()

In [ ]:
ssk = df[df['Блок плагинов'] == "ССК"]

plug_usage = ssk.groupby('username',as_index=False).agg(Кол_во_запусков=('username','count')).sort_values('Кол_во_запусков',ascending=False)
plug_usage = plug_usage.rename({"Кол_во_запусков":"Количество запусков"},axis=1)
plug_usage

In [ ]:

# Создаем фигуру с белым фоном
fig, ax = plt.subplots(figsize=(12, 8), facecolor='white',dpi=600)
ax.set_facecolor('white')

bar_color = '#C1DC8B'

# Создаем barplot с зеленым цветом как на скриншоте
sns.barplot(
    data=plug_usage,
    y='username',
    x='Количество запусков',
    orient='h',
    ax=ax,
    color=bar_color  # Зеленый цвет как на скриншоте (seagreen)
)

# Добавляем подписи на бары (серые)
for container in ax.containers:
    ax.bar_label(
        container,
        fmt='%.0f',
        padding=5,
        color='#666666',  # Серый цвет подписей
        fontsize=10,
        fontweight='normal'
    )

# Добавляем вертикальные серые линии (сетку)
ax.xaxis.grid(
    True,
    which='major',
    color='#E0E0E0',  # Светло-серый цвет линий
    linestyle='-',
    linewidth=0.5,
    alpha=0.7
)
ax.set_axisbelow(True)  # Сетка под барами

# Настраиваем оси
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['left'].set_color('#CCCCCC')
ax.spines['bottom'].set_color('#CCCCCC')

# Добавляем название графика
ax.set_title(
    'Распределение запусков ССК по сотрудникам',
    fontsize=16,
    fontweight='bold',
    pad=20,
    color='#333333'
)

# Настраиваем подписи осей
ax.set_xlabel('Количество запусков', fontsize=12, color='#555555', labelpad=10)
ax.set_ylabel('Сотрудник', fontsize=12, color='#555555', labelpad=10)

# Настраиваем тики
ax.tick_params(axis='both', colors='#666666', labelsize=10)

# Автоматическая подгонка layout
plt.tight_layout()

# Показываем график
plt.show()

In [ ]:
ssk = df[df['Блок плагинов'] == "ClashManager"]

plug_usage = ssk.groupby('username',as_index=False).agg(Кол_во_запусков=('username','count')).sort_values('Кол_во_запусков',ascending=False)
plug_usage = plug_usage.rename({"Кол_во_запусков":"Количество запусков"},axis=1)
plug_usage

In [ ]:

# Создаем фигуру с белым фоном
fig, ax = plt.subplots(figsize=(12, 8), facecolor='white',dpi=600)
ax.set_facecolor('white')

bar_color = '#C1DC8B'

# Создаем barplot с зеленым цветом как на скриншоте
sns.barplot(
    data=plug_usage,
    y='username',
    x='Количество запусков',
    orient='h',
    ax=ax,
    color=bar_color  # Зеленый цвет как на скриншоте (seagreen)
)

# Добавляем подписи на бары (серые)
for container in ax.containers:
    ax.bar_label(
        container,
        fmt='%.0f',
        padding=5,
        color='#666666',  # Серый цвет подписей
        fontsize=10,
        fontweight='normal'
    )

# Добавляем вертикальные серые линии (сетку)
ax.xaxis.grid(
    True,
    which='major',
    color='#E0E0E0',  # Светло-серый цвет линий
    linestyle='-',
    linewidth=0.5,
    alpha=0.7
)
ax.set_axisbelow(True)  # Сетка под барами

# Настраиваем оси
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['left'].set_color('#CCCCCC')
ax.spines['bottom'].set_color('#CCCCCC')

# Добавляем название графика
ax.set_title(
    'Распределение запусков ClashManager по сотрудникам',
    fontsize=16,
    fontweight='bold',
    pad=20,
    color='#333333'
)

# Настраиваем подписи осей
ax.set_xlabel('Количество запусков', fontsize=12, color='#555555', labelpad=10)
ax.set_ylabel('Сотрудник', fontsize=12, color='#555555', labelpad=10)

# Настраиваем тики
ax.tick_params(axis='both', colors='#666666', labelsize=10)

# Автоматическая подгонка layout
plt.tight_layout()

# Показываем график
plt.show()

In [ ]:
ssk = df[df['Блок плагинов'] == "ССК"]

plug_usage = ssk.groupby(['username','event'],as_index=False).agg(Кол_во_запусков=('event','count')).sort_values('Кол_во_запусков',ascending=False)
plug_usage = plug_usage.rename({"Кол_во_запусков":"Количество запусков"},axis=1)
plug_usage['Пользователь_Кнопка'] = plug_usage['username'] + "_" + plug_usage['event'] 
plug_usage

In [ ]:

# Создаем фигуру с белым фоном
fig, ax = plt.subplots(figsize=(12, 8), facecolor='white',dpi=600)
ax.set_facecolor('white')

bar_color = '#C1DC8B'

#оздаем barplot с зеленым цветом как на скриншоте
sns.barplot(
    data=plug_usage,
    y='Пользователь_Кнопка',
    x='Количество запусков',
    orient='h',
    ax=ax,
    color=bar_color  # Зеленый цвет как на скриншоте (seagreen)
)

# Добавляем подписи на бары (серые)
for container in ax.containers:
    ax.bar_label(
        container,
        fmt='%.0f',
        padding=5,
        color='#666666',  # Серый цвет подписей
        fontsize=10,
        fontweight='normal'
    )

# Добавляем вертикальные серые линии (сетку)
ax.xaxis.grid(
    True,
    which='major',
    color='#E0E0E0',  # Светло-серый цвет линий
    linestyle='-',
    linewidth=0.5,
    alpha=0.7
)
ax.set_axisbelow(True)  # Сетка под барами

# Настраиваем оси
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['left'].set_color('#CCCCCC')
ax.spines['bottom'].set_color('#CCCCCC')

# Добавляем название графика
ax.set_title(
    'Распределение запусков ССК по сотрудникам и уровням',
    fontsize=16,
    fontweight='bold',
    pad=20,
    color='#333333'
)

# Настраиваем подписи осей
ax.set_xlabel('Количество запусков', fontsize=12, color='#555555', labelpad=10)
ax.set_ylabel('Сотрудник_кнопка', fontsize=12, color='#555555', labelpad=10)

# Настраиваем тики
ax.tick_params(axis='both', colors='#666666', labelsize=10)

# Автоматическая подгонка layout
plt.tight_layout()

# Показываем график
plt.show()

### Какие кнопки не используются

In [ ]:
# st_date = datetime(2026,2,2)
# end_date = datetime(2026,2,10)
# users = [
#             "a.akulinushkina"
#             ,"a.filimonov"
#             ,"d.usov"
#             ,"e.zapretilin"
#             ,"i.neustroeva"
#             ,"m.bylinkin"
#             ,"p.desyatova"
#             ,"y.kucher"
#             ,"yu.kremenetskaya"
#             ,"d.danovskaya"
#             ,"u.kartasheva"
#             ,"v.semenovych"
#             ,"d.shljapenkova"
#             ,"BIM"
#             ,"n.khabarov"
#             ,"an.goncharuk"
#             ,"m.andreichenko"
#             ,"a.barbakadze"
#             ,"m.nosikov"
#             ,"d.ishmatova"
#             ,"o.erokhova"
#             ]

format = f"%Y-%m-%d"
df = pd.read_csv('plugins_usage.csv')

df['assembly_timestamp'] = df['assembly_timestamp'].apply(lambda x: datetime.strptime(x[:10],format))
df['timestamp'] = df['timestamp'].apply(lambda x: datetime.strptime(x[:10],format))
# df = df[df['timestamp'].between(st_date,end_date)]
# df = df[df['username'].isin(users) == False]
df.head()

In [ ]:
last_plugins = df.groupby('event').agg(
    Кол_во_запусков=('username','count'),
    Крайний_запуск=('timestamp','max')
    )
last_plugins = last_plugins.sort_values('Кол_во_запусков',ascending=True)
last_plugins[last_plugins['Кол_во_запусков'] < 10]

In [ ]:
last_plugins = df.groupby('event').agg(
    Кол_во_запусков=('username','count'),
    Крайний_запуск=('timestamp','max')
    )
last_plugins = last_plugins.sort_values('Крайний_запуск',ascending=True)
last_plugins.to_excel("КрайниеЗапуски.xlsx",sheet_name="Лист1",index=True)